In [ ]:
import pandas as pd

# Load the two compiled lists
ri_storms = pd.read_csv('RI_Storm_List.csv')
non_ri_storms = pd.read_csv('Non_RI_Storm_List.csv')

# Add the Target Label
ri_storms['Target'] = 1
non_ri_storms['Target'] = 0

# Combine into the Master Catalog
full_storm_list = pd.concat([ri_storms, non_ri_storms])
full_storm_list = full_storm_list.sort_values(by='Year', ascending=True, ignore_index=True)

# Display the first few rows to verify the 2001 starts
print(full_storm_list.head())

# Display the last few rows to verify the 2025 additions
print(full_storm_list.tail())

# Save to new csv file:
full_storm_list.to_csv('Full_Storm_List.csv')

In [ ]:
#Test: viewing SST data from era5-single-levels
import cdsapi

dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["sea_surface_temperature"],
    "year": ["2021"],
    "month": ["08"],
    "day": ["19"],
    "time": ["18:00"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [22.5, -91.5, 18.5, -87.5]
}
filename = "test_sst_file.nc"
client = cdsapi.Client()
client.retrieve(dataset, request, filename)

In [ ]:
import xarray as xr

ds = xr.open_dataset(filename)
# Quick check in your processing script
sst_sample = ds['sst'].isel(valid_time=0)
land_pixels = sst_sample.isnull().sum()
total_pixels = sst_sample.size

print(f"Percentage of land/missing pixels in box: {(land_pixels/total_pixels)*100:.2f}%")

In [ ]:
#Test: Obtain RH and shear vector data and find the relevant values:
import cdsapi

dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "relative_humidity",
        "u_component_of_wind",
        "v_component_of_wind"
    ],
    "year": ["2021"],
    "month": ["08"],
    "day": ["19"],
    "time": ["18:00"],
    "pressure_level": [
        "200", "300", "500", "700",
        "850", "800", "1000"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [25.5, -94.5, 15.5, -84.5]
}

client = cdsapi.Client()
filename = 'test_atm_file.nc'
client.retrieve(dataset, request, filename)

In [ ]:
ds1 = xr.open_dataset(filename)
ds1

In [ ]:
ds.close
ds1.close

In [ ]:
#Creation of the storm anchor script

import pandas as pd

# 1. Load your merged catalog (created in the previous step)
master_catalog = pd.read_csv('Full_Storm_List.csv')

def extract_anchors(ibtracs_path, catalog):
    # Load IBTrACS - Skip the 2nd row which contains units
    df = pd.read_csv(ibtracs_path, low_memory=False, skiprows=[1])
    
    # Standardize types and case
    df['SEASON'] = df['SEASON'].astype(int)
    df['NAME'] = df['NAME'].str.upper()
    catalog['Name'] = catalog['Name'].str.upper()
    
    anchors = []

    for index, row in catalog.iterrows():
        # Filter IBTrACS for this specific storm and year
        storm_df = df[(df['NAME'] == row['Name']) & (df['SEASON'] == row['Year'])].copy()
        
        if storm_df.empty:
            print(f"Warning: {row['Name']} ({row['Year']}) not found in IBTrACS.")
            continue
            
        storm_df['ISO_TIME'] = pd.to_datetime(storm_df['ISO_TIME'])
        storm_df = storm_df.sort_values('ISO_TIME')
        
        # Use USA_WIND as the intensity metric
        storm_df['USA_WIND'] = pd.to_numeric(storm_df['USA_WIND'], errors='coerce')
        
        if row['Target'] == 1:
            # RI Logic: Find the FIRST point where V(t+24) - V(t) >= 30
            found_ri = False
            for i in range(len(storm_df)):
                t0_row = storm_df.iloc[i]
                target_time = t0_row['ISO_TIME'] + pd.Timedelta(hours=24)
                
                # Look for the row exactly 24 hours later
                future_row = storm_df[storm_df['ISO_TIME'] == target_time]
                
                if not future_row.empty:
                    v_diff = future_row.iloc[0]['USA_WIND'] - t0_row['USA_WIND']
                    if v_diff >= 30:
                        anchors.append({
                            'Name': row['Name'], 'Year': row['Year'], 'Target': 1,
                            'Anchor_Time': t0_row['ISO_TIME'],
                            'Lat': t0_row['LAT'], 'Lon': t0_row['LON']
                        })
                        found_ri = True
                        break
            if not found_ri:
                print(f"Note: Could not verify 30kt/24hr jump for RI storm {row['Name']}")

        else:
            # Non-RI Logic: 12 hours before first peak intensity (pt)
            if not storm_df['USA_WIND'].dropna().empty:
                max_wind = storm_df['USA_WIND'].max()
                # Get the first instance of peak wind
                pt_row = storm_df[storm_df['USA_WIND'] == max_wind].iloc[0]
                
                anchor_time = pt_row['ISO_TIME'] - pd.Timedelta(hours=12)
                
                # Find the closest available time step at or before pt-12
                anchor_row = storm_df[storm_df['ISO_TIME'] <= anchor_time].tail(1)
                
                if not anchor_row.empty:
                    anchors.append({
                        'Name': row['Name'], 'Year': row['Year'], 'Target': 0,
                        'Anchor_Time': anchor_row.iloc[0]['ISO_TIME'],
                        'Lat': anchor_row.iloc[0]['LAT'], 'Lon': anchor_row.iloc[0]['LON']
                    })

    return pd.DataFrame(anchors)

# 2. Execute Extraction
anchor_df = extract_anchors(r"C:\Users\Sohum Chatterjee\OneDrive\Documents\Programs\Code for Storms\Code-for-storms\ibtracs.ALL.list.v04r01.csv", master_catalog)

# 3. Save the result
anchor_df.to_csv('Storm_Anchors_For_ERA5.csv', index=False)
print(f"Success! Generated {len(anchor_df)} anchor points.")

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import cdsapi
import os
from datetime import timedelta

# 1. SETUP
client = cdsapi.Client()
# Replace with the actual path to your anchor file generated from IBTrACS
anchors = pd.read_csv('Storm_Anchors_For_ERA5.csv')
final_data = []

def get_area(lat, lon):
    """Creates a 5x5 degree box centered on the storm eye"""
    return [lat + 2.5, lon - 2.5, lat - 2.5, lon + 2.5]

# 2. THE PROCESSING LOOP
for i, storm in anchors.iterrows():
    print(f"Processing {storm['Name']} ({storm['Year']})...")
    
    # Filenames for this specific storm
    atm_file = f"temp_atm_{i}.nc"
    sfc_file = f"temp_sfc_{i}.nc"
    
    time_str = pd.to_datetime(storm['Anchor_Time']).strftime('%Y-%m-%d %H:%M')
    area = get_area(storm['Lat'], storm['Lon'])

    try:
        # A. DOWNLOAD ATMOSPHERIC DATA (Pressure Levels)
        client.retrieve("reanalysis-era5-pressure-levels", {
            "product_type": "reanalysis",
            "variable": ["u_component_of_wind", "v_component_of_wind", "relative_humidity"],
            "pressure_level": ["200", "300", "500", "800", "850", "1000"],
            "year": str(storm['Year']),
            "month": str(pd.to_datetime(storm['Anchor_Time']).month).zfill(2),
            "day": str(pd.to_datetime(storm['Anchor_Time']).day).zfill(2),
            "time": pd.to_datetime(storm['Anchor_Time']).strftime('%H:%M'),
            "area": area,
            "format": "netcdf",
        }, atm_file)

        # B. DOWNLOAD OCEAN DATA (Single Levels)
        client.retrieve("reanalysis-era5-single-levels", {
            "product_type": "reanalysis",
            "variable": "sea_surface_temperature",
            "year": str(storm['Year']),
            "month": str(pd.to_datetime(storm['Anchor_Time']).month).zfill(2),
            "day": str(pd.to_datetime(storm['Anchor_Time']).day).zfill(2),
            "time": pd.to_datetime(storm['Anchor_Time']).strftime('%H:%M'),
            "area": area,
            "format": "netcdf",
        }, sfc_file)

        # C. FEATURE EXTRACTION (Xarray)
        ds_atm = xr.open_dataset(atm_file)
        ds_sfc = xr.open_dataset(sfc_file)

        # Calculate Shear Pairs
        shear_layers = {
            "shear_200_850": (200, 850),
            "shear_500_850": (500, 850),
            "shear_200_500": (200, 500),
            "shear_300_800": (300, 800),
            "shear_850_1000": (850, 1000),
        }
        
        row_results = {
            "Name": storm['Name'],
            "Year": storm['Year'],
            "Target": storm['Target']
        }

        # Calculate Shear Magnitudes (Area Mean)
        for key, (top, bottom) in shear_layers.items():
            u_diff = ds_atm.u.sel(pressure_level=top) - ds_atm.u.sel(pressure_level=bottom)
            v_diff = ds_atm.v.sel(pressure_level=top) - ds_atm.v.sel(pressure_level=bottom)
            mag = np.sqrt(u_diff**2 + v_diff**2)
            row_results[key] = float(mag.mean()) * 1.94384 #Store the result in knots....

        # RH Avg (Mid-level focus: 500-850 hPa)
        row_results["RH_avg"] = float(ds_atm.r.sel(pressure_level=[500, 800, 850]).mean())

        # SST Avg (NaN-aware for water-only)
        row_results["SST_avg"] = float(ds_sfc.sst.mean(skipna=True))

        #Debug: Print the data for checking purposes
        print(row_results)
        final_data.append(row_results)

        # D. CLEANUP
        ds_atm.close()
        ds_sfc.close()
        os.remove(atm_file)
        os.remove(sfc_file)

    except Exception as e:
        print(f"Error processing {storm['Name']}: {e}")

# 3. SAVE FINAL DATASET
df_final = pd.DataFrame(final_data)
df_final.to_csv('Final_TC_RI_Dataset.csv', index=False)
print("Project Dataset Complete!")

Processing CHANTAL (2001)...


2026-04-03 20:31:57,775 INFO Request ID is 6a32d48a-3bee-435e-a076-1adb995ca56c
2026-04-03 20:31:57,980 INFO status has been updated to accepted
2026-04-03 20:32:12,267 INFO status has been updated to running
2026-04-03 20:32:20,071 INFO status has been updated to successful


7df10ad34aabb393ec7881037b8b588c.nc:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

2026-04-03 20:32:24,586 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-03 20:32:24,586 INFO Request ID is 73f669cb-c4e7-4192-b88e-10d24346c613
2026-04-03 20:32:24,797 INFO status has been updated to accepted
2026-04-03 20:32:39,105 INFO status has been updated to successful


da443f99eece7f6b8c9eecb7940ea48b.nc:   0%|          | 0.00/24.6k [00:00<?, ?B/s]

...

2026-04-04 01:12:33,075 INFO Request ID is 292a884d-9abd-4a22-bfd0-2643ca721a68
2026-04-04 01:12:34,318 INFO status has been updated to accepted
2026-04-04 01:12:48,709 INFO status has been updated to running
2026-04-04 01:13:08,207 INFO status has been updated to successful


ccbf03cc0934fd4c744223e397aa428b.nc:   0%|          | 0.00/65.6k [00:00<?, ?B/s]

2026-04-04 01:13:11,709 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-04 01:13:11,710 INFO Request ID is 325bfcc6-c4d7-4e95-aa34-8910dbbf2400
2026-04-04 01:13:11,942 INFO status has been updated to accepted
2026-04-04 01:13:26,128 INFO status has been updated to running
2026-04-04 01:13:33,971 INFO status has been updated to successful


ab63544ce27800970b89e102f7172b82.nc:   0%|          | 0.00/24.6k [00:00<?, ?B/s]

{'Name': 'ERIN', 'Year': 2025, 'Target': 1, 'shear_200_850': 24.903540596008302, 'shear_500_850': 8.807169475784303, 'shear_200_500': 21.06647648590088, 'shear_300_800': 16.432788638763427, 'shear_850_1000': 7.1255434768676755, 'RH_avg': 75.44425201416016, 'SST_avg': 301.4358215332031}
Project Dataset Complete!


In [2]:
#Some post processing....
final_dataset = pd.read_csv('Final_TC_RI_Dataset.csv')

final_dataset['SST_avg'] = final_dataset['SST_avg'] - 273.15

final_dataset.to_csv('Final_TC_RI_Dataset.csv', index=False)

In [3]:
import pandas as pd

# Load the two compiled lists
ri_storms = pd.read_csv('Extra_RI.csv')
non_ri_storms = pd.read_csv('Extra_Non_RI.csv')

# Add the Target Label
ri_storms['Target'] = 1
non_ri_storms['Target'] = 0

# Combine into the Master Catalog
full_storm_list = pd.concat([ri_storms, non_ri_storms])
full_storm_list = full_storm_list.sort_values(by='Year', ascending=True, ignore_index=True)

# Display the first few rows to verify the 2001 starts
print(full_storm_list.head())

# Display the last few rows to verify the 2025 additions
print(full_storm_list.tail())

# Save to new csv file:
full_storm_list.to_csv('Extra_Storm_List.csv')

      Name  Year  Target
0  Malakas  2010       1
1     Megi  2010       1
2   Fanapi  2010       0
      Name  Year  Target
0  Malakas  2010       1
1     Megi  2010       1
2   Fanapi  2010       0


In [4]:
#Creation of the storm anchor script

import pandas as pd

# 1. Load your merged catalog (created in the previous step)
master_catalog = pd.read_csv('Extra_Storm_List.csv')

def extract_anchors(ibtracs_path, catalog):
    # Load IBTrACS - Skip the 2nd row which contains units
    df = pd.read_csv(ibtracs_path, low_memory=False, skiprows=[1])
    
    # Standardize types and case
    df['SEASON'] = df['SEASON'].astype(int)
    df['NAME'] = df['NAME'].str.upper()
    catalog['Name'] = catalog['Name'].str.upper()
    
    anchors = []

    for index, row in catalog.iterrows():
        # Filter IBTrACS for this specific storm and year
        storm_df = df[(df['NAME'] == row['Name']) & (df['SEASON'] == row['Year'])].copy()
        
        if storm_df.empty:
            print(f"Warning: {row['Name']} ({row['Year']}) not found in IBTrACS.")
            continue
            
        storm_df['ISO_TIME'] = pd.to_datetime(storm_df['ISO_TIME'])
        storm_df = storm_df.sort_values('ISO_TIME')
        
        # Use USA_WIND as the intensity metric
        storm_df['USA_WIND'] = pd.to_numeric(storm_df['USA_WIND'], errors='coerce')
        
        if row['Target'] == 1:
            # RI Logic: Find the FIRST point where V(t+24) - V(t) >= 30
            found_ri = False
            for i in range(len(storm_df)):
                t0_row = storm_df.iloc[i]
                target_time = t0_row['ISO_TIME'] + pd.Timedelta(hours=24)
                
                # Look for the row exactly 24 hours later
                future_row = storm_df[storm_df['ISO_TIME'] == target_time]
                
                if not future_row.empty:
                    v_diff = future_row.iloc[0]['USA_WIND'] - t0_row['USA_WIND']
                    if v_diff >= 30:
                        anchors.append({
                            'Name': row['Name'], 'Year': row['Year'], 'Target': 1,
                            'Anchor_Time': t0_row['ISO_TIME'],
                            'Lat': t0_row['LAT'], 'Lon': t0_row['LON']
                        })
                        found_ri = True
                        break
            if not found_ri:
                print(f"Note: Could not verify 30kt/24hr jump for RI storm {row['Name']}")

        else:
            # Non-RI Logic: 12 hours before first peak intensity (pt)
            if not storm_df['USA_WIND'].dropna().empty:
                max_wind = storm_df['USA_WIND'].max()
                # Get the first instance of peak wind
                pt_row = storm_df[storm_df['USA_WIND'] == max_wind].iloc[0]
                
                anchor_time = pt_row['ISO_TIME'] - pd.Timedelta(hours=12)
                
                # Find the closest available time step at or before pt-12
                anchor_row = storm_df[storm_df['ISO_TIME'] <= anchor_time].tail(1)
                
                if not anchor_row.empty:
                    anchors.append({
                        'Name': row['Name'], 'Year': row['Year'], 'Target': 0,
                        'Anchor_Time': anchor_row.iloc[0]['ISO_TIME'],
                        'Lat': anchor_row.iloc[0]['LAT'], 'Lon': anchor_row.iloc[0]['LON']
                    })

    return pd.DataFrame(anchors)

# 2. Execute Extraction
anchor_df = extract_anchors(r"C:\Users\Sohum Chatterjee\OneDrive\Documents\Programs\Code for Storms\Code-for-storms\ibtracs.ALL.list.v04r01.csv", master_catalog)

# 3. Save the result
anchor_df.to_csv('Extra_Storm_Anchors_For_ERA5.csv', index=False)
print(f"Success! Generated {len(anchor_df)} anchor points.")

Success! Generated 3 anchor points.


In [5]:
import pandas as pd
import xarray as xr
import numpy as np
import cdsapi
import os
from datetime import timedelta

# 1. SETUP
client = cdsapi.Client()
# Replace with the actual path to your anchor file generated from IBTrACS
anchors = pd.read_csv('Extra_Storm_Anchors_For_ERA5.csv')
final_data = []

def get_area(lat, lon):
    """Creates a 5x5 degree box centered on the storm eye"""
    return [lat + 2.5, lon - 2.5, lat - 2.5, lon + 2.5]

# 2. THE PROCESSING LOOP
for i, storm in anchors.iterrows():
    print(f"Processing {storm['Name']} ({storm['Year']})...")
    
    # Filenames for this specific storm
    atm_file = f"temp_atm_{i}.nc"
    sfc_file = f"temp_sfc_{i}.nc"
    
    time_str = pd.to_datetime(storm['Anchor_Time']).strftime('%Y-%m-%d %H:%M')
    area = get_area(storm['Lat'], storm['Lon'])

    try:
        # A. DOWNLOAD ATMOSPHERIC DATA (Pressure Levels)
        client.retrieve("reanalysis-era5-pressure-levels", {
            "product_type": "reanalysis",
            "variable": ["u_component_of_wind", "v_component_of_wind", "relative_humidity"],
            "pressure_level": ["200", "300", "500", "800", "850", "1000"],
            "year": str(storm['Year']),
            "month": str(pd.to_datetime(storm['Anchor_Time']).month).zfill(2),
            "day": str(pd.to_datetime(storm['Anchor_Time']).day).zfill(2),
            "time": pd.to_datetime(storm['Anchor_Time']).strftime('%H:%M'),
            "area": area,
            "format": "netcdf",
        }, atm_file)

        # B. DOWNLOAD OCEAN DATA (Single Levels)
        client.retrieve("reanalysis-era5-single-levels", {
            "product_type": "reanalysis",
            "variable": "sea_surface_temperature",
            "year": str(storm['Year']),
            "month": str(pd.to_datetime(storm['Anchor_Time']).month).zfill(2),
            "day": str(pd.to_datetime(storm['Anchor_Time']).day).zfill(2),
            "time": pd.to_datetime(storm['Anchor_Time']).strftime('%H:%M'),
            "area": area,
            "format": "netcdf",
        }, sfc_file)

        # C. FEATURE EXTRACTION (Xarray)
        ds_atm = xr.open_dataset(atm_file)
        ds_sfc = xr.open_dataset(sfc_file)

        # Calculate Shear Pairs
        shear_layers = {
            "shear_200_850": (200, 850),
            "shear_500_850": (500, 850),
            "shear_200_500": (200, 500),
            "shear_300_800": (300, 800),
            "shear_850_1000": (850, 1000),
        }
        
        row_results = {
            "Name": storm['Name'],
            "Year": storm['Year'],
            "Target": storm['Target']
        }

        # Calculate Shear Magnitudes (Area Mean)
        for key, (top, bottom) in shear_layers.items():
            u_diff = ds_atm.u.sel(pressure_level=top) - ds_atm.u.sel(pressure_level=bottom)
            v_diff = ds_atm.v.sel(pressure_level=top) - ds_atm.v.sel(pressure_level=bottom)
            mag = np.sqrt(u_diff**2 + v_diff**2)
            row_results[key] = float(mag.mean()) * 1.94384 #Store the result in knots....

        # RH Avg (Mid-level focus: 500-850 hPa)
        row_results["RH_avg"] = float(ds_atm.r.sel(pressure_level=[500, 800, 850]).mean())

        # SST Avg (NaN-aware for water-only)
        row_results["SST_avg"] = float(ds_sfc.sst.mean(skipna=True))

        #Debug: Print the data for checking purposes
        print(row_results)
        final_data.append(row_results)

        # D. CLEANUP
        ds_atm.close()
        ds_sfc.close()
        os.remove(atm_file)
        os.remove(sfc_file)

    except Exception as e:
        print(f"Error processing {storm['Name']}: {e}")

# 3. SAVE FINAL DATASET
df_final = pd.DataFrame(final_data)
df_final.to_csv('Extra_TC_RI_Dataset.csv', index=False)
print("Extra Dataset Complete!")

Processing MALAKAS (2010)...


2026-04-04 01:44:53,917 INFO Request ID is 396a5e8f-d06b-4be2-88fc-abe7ffbc30f9
2026-04-04 01:44:54,123 INFO status has been updated to accepted
2026-04-04 01:45:03,184 INFO status has been updated to running
2026-04-04 01:45:08,493 INFO status has been updated to successful


f0e9ac9371fec3d4fd5ea37011adb213.nc:   0%|          | 0.00/65.2k [00:00<?, ?B/s]

2026-04-04 01:45:11,709 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-04 01:45:11,710 INFO Request ID is e2f5918e-f0b2-4224-b2aa-ba48ccca2f09
2026-04-04 01:45:11,921 INFO status has been updated to accepted
2026-04-04 01:45:26,429 INFO status has been updated to running
2026-04-04 01:45:34,315 INFO status has been updated to successful


93c6deedb782aae4077eacabea3d2400.nc:   0%|          | 0.00/24.6k [00:00<?, ?B/s]

...

2026-04-04 01:46:57,246 INFO Request ID is 9d20db83-6734-4bef-8a95-629f95774bfc
2026-04-04 01:46:57,465 INFO status has been updated to accepted
2026-04-04 01:47:11,755 INFO status has been updated to successful


a5a85c8d993244d07c8ddc38ea48d725.nc:   0%|          | 0.00/64.8k [00:00<?, ?B/s]

2026-04-04 01:47:14,975 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-04 01:47:14,976 INFO Request ID is 121041b3-6ef9-4804-a736-d2f6a26bbb78
2026-04-04 01:47:15,185 INFO status has been updated to accepted
2026-04-04 01:47:24,191 INFO status has been updated to running
2026-04-04 01:47:29,457 INFO status has been updated to successful


ab6948237fcb36743a2c86cd88b8b296.nc:   0%|          | 0.00/24.6k [00:00<?, ?B/s]

{'Name': 'FANAPI', 'Year': 2010, 'Target': 0, 'shear_200_850': 31.027103009643554, 'shear_500_850': 15.66770158203125, 'shear_200_500': 26.87011918762207, 'shear_300_800': 20.323093605804445, 'shear_850_1000': 19.36930923614502, 'RH_avg': 83.69011688232422, 'SST_avg': 302.1367492675781}
Extra Dataset Complete!


In [8]:
final1 = pd.read_csv('Final_TC_RI_Dataset.csv')
extra1 = pd.read_csv('Extra_TC_RI_Dataset.csv')

extra1['SST_avg'] = extra1['SST_avg'] - 273.15

final1 = pd.concat([final1, extra1], ignore_index=True)
final1 = final1.sort_values(by='Year', ascending=True, ignore_index=True)

final1.to_csv('Final_TC_RI_Dataset.csv', index=False)

In [1]:
import pandas as pd

# 1. Load the two compiled anchor files
main_anchors = pd.read_csv('Storm_Anchors_For_ERA5.csv')
extra_anchors = pd.read_csv('Extra_Storm_Anchors_For_ERA5.csv')

# 2. Combine into one large DataFrame
# ignore_index=True re-indexes the file from 0 to the final total count
combined_anchors = pd.concat([main_anchors, extra_anchors], ignore_index=True)

# 3. Final cleanup: Ensure all Names are uppercase for consistency
combined_anchors['Name'] = combined_anchors['Name'].str.upper()

# 4. Save the combined master list
combined_anchors.to_csv('Combined_Storm_Anchors_For_ERA5.csv', index=False)

# 5. Summary verification
print(f"Main list count: {len(main_anchors)}")
print(f"Extra list count: {len(extra_anchors)}")
print(f"Combined total: {len(combined_anchors)} storms ready for ERA5.")

Main list count: 334
Extra list count: 3
Combined total: 337 storms ready for ERA5.


In [2]:
import pandas as pd
import numpy as np

def update_dataset_with_roci(final_csv, anchor_csv, ibtracs_path):
    # 1. Load the existing datasets
    df_final = pd.read_csv(final_csv)
    df_anchors = pd.read_csv(anchor_csv)
    
    # Load IBTrACS (Skipping the units row)
    ib_df = pd.read_csv(ibtracs_path, low_memory=False, skiprows=[1])
    
    # 2. Standardize Names and Times
    df_anchors['Anchor_Time'] = pd.to_datetime(df_anchors['Anchor_Time'])
    ib_df['ISO_TIME'] = pd.to_datetime(ib_df['ISO_TIME'])
    ib_df['NAME'] = ib_df['NAME'].str.upper()
    df_final['Name'] = df_final['Name'].str.upper()
    df_anchors['Name'] = df_anchors['Name'].str.upper()

    roci_values = []

    # 3. The "Catching" Loop
    for i, row in df_anchors.iterrows():
        storm_name = row['Name']
        storm_year = int(row['Year'])
        anchor_t = row['Anchor_Time']
        
        # Filter IBTrACS for this specific storm lifecycle
        storm_track = ib_df[(ib_df['NAME'] == storm_name) & 
                            (ib_df['SEASON'] == storm_year)].copy()
        
        if storm_track.empty:
            roci_values.append(np.nan)
            continue
        
        # Convert ROCI to numeric (using USA_ROCI as it's the most complete field)
        storm_track['ROCI'] = pd.to_numeric(storm_track['USA_ROCI'], errors='coerce')
        
        # Try to find the exact anchor time value first
        exact_match = storm_track[storm_track['ISO_TIME'] == anchor_t]
        
        val = np.nan
        if not exact_match.empty and not pd.isna(exact_match['ROCI'].iloc[0]):
            val = exact_match['ROCI'].iloc[0]
        else:
            # --- THE CATCHING SCRIPT ---
            # If exact is empty, look for the nearest valid ROCI in the storm's history
            valid_roci_points = storm_track.dropna(subset=['ROCI'])
            
            if not valid_roci_points.empty:
                # Calculate time difference to find the absolute closest point
                valid_roci_points['time_diff'] = (valid_roci_points['ISO_TIME'] - anchor_t).abs()
                closest_point = valid_roci_points.sort_values('time_diff').iloc[0]
                
                # Check if the closest point is within a reasonable window (e.g., 24 hours)
                if closest_point['time_diff'] <= pd.Timedelta(hours=24):
                    val = closest_point['ROCI']
                else:
                    # If too far, interpolate if possible, or leave as NaN
                    val = np.nan
        
        roci_values.append(val)

    # 4. Add the column to the dataset
    # We map it back to df_final based on Name and Year to ensure alignment
    df_final['ROCI'] = roci_values
    
    # 5. Final Cleaning
    # Optional: Fill remaining NaNs with the median ROCI of the dataset 
    # (keeps the model from crashing while maintaining statistical neutral)
    # df_final['ROCI'] = df_final['ROCI'].fillna(df_final['ROCI'].median())

    df_final.to_csv('Final_TC_RI_Dataset.csv', index=False)
    print(f"Success! Column added. Valid ROCI found for {df_final['ROCI'].notnull().sum()} out of {len(df_final)} storms.")

# Run the function
update_dataset_with_roci('Final_TC_RI_Dataset.csv', 
                        'Combined_Storm_Anchors_For_ERA5.csv', 
                        r"C:\Users\Sohum Chatterjee\OneDrive\Documents\Programs\Code for Storms\Code-for-storms\ibtracs.ALL.list.v04r01.csv")

C:\Users\Sohum Chatterjee\AppData\Local\Temp\ipykernel_13208\4049545975.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_roci_points['time_diff'] = (valid_roci_points['ISO_TIME'] - anchor_t).abs()
C:\Users\Sohum Chatterjee\AppData\Local\Temp\ipykernel_13208\4049545975.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_roci_points['time_diff'] = (valid_roci_points['ISO_TIME'] - anchor_t).abs()
C:\Users\Sohum Chatterjee\AppData\Local\Temp\ipykernel_13208\4049545975.py:51: SettingWithCopyWarn

Success! Column added. Valid ROCI found for 327 out of 337 storms.


In [3]:
import pandas as pd
import numpy as np

def add_anchor_winds(final_dataset_path, anchor_list_path, ibtracs_path):
    # 1. Load the existing datasets
    print("Loading datasets...")
    df_final = pd.read_csv(final_dataset_path)
    df_anchors = pd.read_csv(anchor_list_path)
    
    # Load IBTrACS - skipping the units row (2nd row)
    # Using low_memory=False to handle mixed types in IBTrACS columns
    ib_df = pd.read_csv(ibtracs_path, low_memory=False, skiprows=[1])
    
    # 2. Standardize data types and casing
    print("Standardizing formats...")
    df_anchors['Anchor_Time'] = pd.to_datetime(df_anchors['Anchor_Time'])
    ib_df['ISO_TIME'] = pd.to_datetime(ib_df['ISO_TIME'])
    
    # Ensure Name and Year columns are consistent
    ib_df['NAME'] = ib_df['NAME'].str.upper()
    ib_df['SEASON'] = ib_df['SEASON'].astype(int)
    df_anchors['Name'] = df_anchors['Name'].str.upper()
    df_anchors['Year'] = df_anchors['Year'].astype(int)
    
    # Use WMO_WIND or USA_WIND (intensity in knots)
    # WMO_WIND is the standard international estimate
    ib_df['WMO_WIND'] = pd.to_numeric(ib_df['WMO_WIND'], errors='coerce')

    # 3. Create a mapping dictionary for fast lookup
    # Key: (Name, Year, ISO_TIME) -> Value: WMO_WIND
    print("Building wind lookup map...")
    wind_map = ib_df.set_index(['NAME', 'SEASON', 'ISO_TIME'])['WMO_WIND'].to_dict()

    # 4. Extract winds for each anchor point
    print("Matching winds to anchor points...")
    anchor_winds = []
    for i, row in df_anchors.iterrows():
        key = (row['Name'], row['Year'], row['Anchor_Time'])
        wind = wind_map.get(key, np.nan)
        anchor_winds.append(wind)
    
    # Add the column to the anchors dataframe
    df_anchors['v_0'] = anchor_winds

    # 5. Merge the Wind column into the Final Dataset
    # We merge on Name, Year, and Target to ensure everything aligns perfectly
    print("Merging into final dataset...")
    # Select only the key columns and the new feature from anchors
    df_winds_only = df_anchors[['Name', 'Year', 'Target', 'v_0']]
    
    df_merged = pd.merge(df_final, df_winds_only, on=['Name', 'Year', 'Target'], how='left')

    # 6. Save the results
    output_path = 'Final_TC_RI_Dataset.csv'
    df_merged.to_csv(output_path, index=False)
    
    print(f"\nSuccess!")
    print(f"File saved: {output_path}")
    print(f"Total Rows: {len(df_merged)}")
    print(f"Valid Wind values: {df_merged['v_0'].notnull().sum()}")

# Execute the script
if __name__ == "__main__":
    add_anchor_winds(
        final_dataset_path='Final_TC_RI_Dataset.csv',
        anchor_list_path='Combined_Storm_Anchors_For_ERA5.csv',
        ibtracs_path=r"C:\Users\Sohum Chatterjee\OneDrive\Documents\Programs\Code for Storms\Code-for-storms\ibtracs.ALL.list.v04r01.csv"
    )

Loading datasets...
Standardizing formats...
Building wind lookup map...
Matching winds to anchor points...
Merging into final dataset...

Success!
File saved: Final_TC_RI_Dataset.csv
Total Rows: 339
Valid Wind values: 288


In [4]:
import pandas as pd
import numpy as np
import math

def calculate_rmw_nm(v_0_kt, lat, roci_nm):
    """
    Calculates the Radius of Maximum Winds (RMW) based on 
    Vmax, Latitude, and ROCI using a momentum-ratio approach.
    """
    # Handle missing values
    if pd.isna(v_0_kt) or pd.isna(lat) or pd.isna(roci_nm) or roci_nm <= 0:
        return np.nan

    try:
        # 1. Preprocessing & Unit Conversion
        # Convert ROCI from nautical miles to meters
        roci_m = roci_nm * 1.852 * 1000 
        
        # Coriolis Parameter (f)
        f = 2 * (7.292 * 10**(-5)) * math.sin(math.radians(abs(lat)))
        
        # Convert 1-min peak wind (knots) to 10-min equivalent and then to m/s
        vmax_ms = (v_0_kt * 0.93) / 1.944 
        
        # Avoid division by zero at the equator
        if f == 0: 
            return np.nan

        # 2. Step 1: Calculate Angular Momentum at ROCI (M_roci)
        # Assumes velocity at ROCI (v_roci) is roughly 11.39 m/s
        v_roci = 11.39
        M_roci = (roci_m * v_roci) + (0.5 * f * (roci_m ** 2))

        # 3. Step 2: Calculate Momentum Ratio
        # Constants derived from statistical TC structural relationships
        b, beta_1, beta_2 = 0.24, -0.0030281, -0.00080726
        
        # Momentum ratio defines how angular momentum is distributed from eye to ROCI
        exp_term = (beta_1 * (vmax_ms - v_roci)) + (beta_2 * (vmax_ms - v_roci) * ((f * roci_m) / 2))
        momentum_ratio = b * math.exp(exp_term)

        # 4. Step 3: Calculate Momentum at RMW (M_rmw)
        M_rmw = momentum_ratio * M_roci

        # 5. Step 4: Solve for RMW (R_rmw) in meters
        # Based on the modified Rankine vortex or similar parametric profile
        term_under_sqrt = 1 + (2 * f * M_rmw) / (vmax_ms**2)
        R_rmw_m = (vmax_ms / f) * (math.sqrt(term_under_sqrt) - 1)

        # 6. Convert final result back to Nautical Miles
        return R_rmw_m / 1852

    except (ValueError, ZeroDivisionError):
        return np.nan

# --- Main Script ---

# 1. Load the inputs
df_final = pd.read_csv('Final_TC_RI_Dataset.csv')
df_anchors = pd.read_csv('Combined_Storm_Anchors_For_ERA5.csv')

# 2. Merge Latitude from the anchors into the final dataset
# We merge on Name, Year, and Target to ensure rows match correctly
df_merged = pd.merge(
    df_final, 
    df_anchors[['Name', 'Year', 'Target', 'Lat']], 
    on=['Name', 'Year', 'Target'], 
    how='left'
)

# 3. Apply the RMW calculation
print("Calculating RMW for each storm case...")
df_merged['RMW'] = df_merged.apply(
    lambda row: calculate_rmw_nm(row['v_0'], row['Lat'], row['ROCI']), 
    axis=1
)

# 4. Cleanup: Remove the temporary 'Lat' column if you don't want it in final output
# df_merged = df_merged.drop(columns=['Lat'])

# 5. Save the updated dataset
output_file = 'Final_TC_RI_Dataset.csv'
df_merged.to_csv(output_file, index=False)

print(f"Success! Added RMW column to {output_file}")
print(f"Calculated valid RMW for {df_merged['RMW'].notnull().sum()} cases.")

Calculating RMW for each storm case...
Success! Added RMW column to Final_TC_RI_Dataset.csv
Calculated valid RMW for 337 cases.
